In [7]:
import pandas as pd
import joblib
import time 

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import re

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

import os


### Load models

In [8]:
# Note: _sm stands for a model trained on SMOTE-enhanced data

# Logistic Regression
lr = joblib.load('models/logistic_regression_best.pkl')
lr_sm = joblib.load('models/logistic_regression_smote_best.pkl')

# Decision Tree
dec_tree = joblib.load('models/decision_tree_best.pkl')
dec_tree_sm = joblib.load('models/decision_tree_smote_best.pkl')

# Random Forest
rf = joblib.load('models/best_rf.pkl')
rf_sm = joblib.load('models/best_rf.pkl')

# XGBoost
xgb = joblib.load('models/best_xgb.pkl')
xgb_sm = joblib.load('models/best_xgb_sm.pkl')

# LightGBM
lgbm = joblib.load('models/best_lightgbm.pkl')
lgbm_sm = joblib.load('models/best_lightgbm_sm.pkl')

In [9]:
# Store models in two lists (because of the different train and test datasets)

lr_models = {
    "Logistic Regression": lr, 
    "Logistic Regression SMOTE": lr_sm
}

non_lr_models = {
    "Decision Tree": dec_tree,
    "Decision Tree SMOTE": dec_tree_sm,
    "Random Forest": rf,
    "Random Forest SMOTE": rf_sm,
    "XGBoost": xgb,
    "XGBoost SMOTE": xgb_sm,
    "LightGBM": lgbm,
    "LightGBM SMOTE": lgbm_sm
}

### Store all thresholds in one place
Currently, the thresholds for every model are stored in different notebooks. To correct this, they will now be stored in a single dictionary and a .csv file

In [10]:
thresholds = {
    "Logistic Regression": 0.16,
    "Logistic Regression SMOTE": 0.68, 
    "Decision Tree": 0.6,
    "Decision Tree SMOTE": 0.05,
    "Random Forest": 0.5,
    "Random Forest SMOTE": 0.15,
    "XGBoost": 0.45,
    "XGBoost SMOTE": 0.15,
    "LightGBM": 0.6,
    "LightGBM SMOTE": 0.15,
}

In [11]:
thresholds_df = pd.DataFrame(list(thresholds.items()), columns=['model', 'threshold'])
thresholds_df.to_csv('model_results/thresholds.csv')

### Load train and test datasets for quick comparison 


In [12]:
X_train_lr = pd.read_parquet('X_train_lr.parquet')
y_train_lr = pd.read_parquet('y_train_lr.parquet')
X_train_lr_smote = pd.read_parquet('X_train_lr_smote.parquet')
y_train_lr_smote = pd.read_parquet('y_train_lr_smote.parquet')
X_val_lr = pd.read_parquet('X_val_lr.parquet')
y_val_lr = pd.read_parquet('y_val_lr.parquet')
X_test_lr = pd.read_parquet('X_test_lr.parquet')
y_test_lr = pd.read_parquet('y_test_lr.parquet')

X_train_others = pd.read_parquet('X_train_others.parquet')
y_train_others = pd.read_parquet('y_train_others.parquet')
X_train_others_smote = pd.read_parquet('X_train_others_smote.parquet')
y_train_others_smote = pd.read_parquet('y_train_others_smote.parquet')
X_val_others = pd.read_parquet('X_val_others.parquet')
y_val_others = pd.read_parquet('y_val_others.parquet')
X_test_others = pd.read_parquet('X_test_others.parquet')
y_test_others = pd.read_parquet('y_test_others.parquet')

### Evaluate models

#### Note: All models will be refitted so that the training time can be calculated

In [13]:
# Create a function for model evaluation 
def evaluate_model(
    best_model,
    X_test,
    y_test,
    threshold,
    training_time,
    model_name
):
    """
    Evaluates a trained model and returns all relevant metrics.
    """

    # -----------------------------
    # Prediction time
    # -----------------------------
    start = time.time()

    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    prediction_time = time.time() - start

    # -----------------------------
    # Metrics
    # -----------------------------
    results = {
        "Model": model_name,
        "Threshold": threshold,

        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob),

        "Training Time (s)": training_time,
        "Prediction Time (s)": prediction_time,

        "Hyperparameters": best_model.get_params(),

        "Confusion Matrix": confusion_matrix(y_test, y_pred),

        "Classification Report": classification_report(
            y_test,
            y_pred,
            output_dict=True
        )
    }

    return results

In [14]:
# Store all model results in a list
all_results = []

In [15]:
# First, do for lr models

for model_name, model in lr_models.items():

    start = time.time()

    # Differentiate models fitted on SMOTE and non-SMOTE data
    if model_name.endswith('SMOTE'):    # If on SMOTE data
        model.fit(X_train_lr_smote, y_train_lr_smote)

    else:   #If not on SMOTE data
        model.fit(X_train_lr, y_train_lr)
    training_time = time.time() - start

    results = evaluate_model(
    best_model=model,
    X_test=X_test_lr,
    y_test=y_test_lr,
    threshold=thresholds[model_name],
    training_time=training_time,
    model_name=model_name
    )


    # Store the model results
    all_results.append(results)


c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid t

In [16]:
# Do the same for non-lr models

for model_name, model in non_lr_models.items():

    start = time.time()
    # Differentiate models fitted on SMOTE and non-SMOTE data
    if model_name.endswith('SMOTE'):    # If on SMOTE data
        #LightGBM returns an error due to column names. Thus, they must be separately renamed:
        if model_name.startswith('LightGBM'):   # IF LightGBM

            # Create a mapping from old to new names
            new_columns = [
                re.sub(r'[^A-Za-z0-9_]', '_', col)
                for col in X_train_others_smote.columns
            ]

            # Rename both datasets identically to solve the JSON problem of LightGBM
            X_train_others_LGBM_sm = X_train_others_smote.copy()
            X_test_others_LGBM = X_test_others.copy()

            X_train_others_LGBM_sm.columns = new_columns
            X_test_others_LGBM.columns = new_columns

            model.fit(X_train_others_LGBM_sm, y_train_others_smote)
        else:
            model.fit(X_train_others_smote, y_train_others_smote)

    else: #If not on SMOTE
        #LightGBM returns an error due to column names. Thus, they must be separately renamed:
        if model_name.startswith('LightGBM'):   # If LightGBM

            # Create a mapping from old to new names
            new_columns = [
                re.sub(r'[^A-Za-z0-9_]', '_', col)
                for col in X_train_others_smote.columns
            ]

            # Rename both datasets identically
            X_train_others_LGBM = X_train_others.copy()
            X_test_others_LGBM = X_test_others.copy()

            X_train_others_LGBM.columns = new_columns
            X_test_others_LGBM.columns = new_columns

            model.fit(X_train_others_LGBM, y_train_others)
        else:
            model.fit(X_train_others, y_train_others)
    training_time = time.time() - start

    if model_name.startswith('LightGBM'):
        results = evaluate_model(
        best_model=model,
        X_test=X_test_others_LGBM,
        y_test=y_test_others,
        threshold=thresholds[model_name],
        training_time=training_time,
        model_name=model_name
        )
    else:
        results = evaluate_model(
        best_model=model,
        X_test=X_test_others,
        y_test=y_test_others,
        threshold=thresholds[model_name],
        training_time=training_time,
        model_name=model_name
        )


    # Store the model results
    all_results.append(results)


c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\core.py:553: UserWarning: [09:18:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutio

In [17]:
summary = pd.DataFrame([{
    "Model": r["Model"],
    "Threshold": r["Threshold"],
    "Accuracy": r["Accuracy"],
    "Precision": r["Precision"],
    "Recall": r["Recall"],
    "F1": r["F1"],
    "ROC_AUC": r["ROC_AUC"],
    "Training Time (s)": r["Training Time (s)"],
    "Prediction Time (s)": r["Prediction Time (s)"]
} for r in all_results])

# Round to 4 decimal places
summary = summary.round(4)
summary

,Model,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,Training Time (s),Prediction Time (s)
0,Logistic Regression,0.16,0.8578,0.2443,0.3637,0.2923,0.7471,2.2200,0.0368
1,Logistic Regression SMOTE,0.68,0.8400,0.2265,0.4066,0.2910,0.7411,5.5224,0.0501
2,Decision Tree,0.60,0.7734,0.1763,0.4918,0.2595,0.7089,8.4661,0.0319
3,Decision Tree SMOTE,0.05,0.7966,0.1332,0.2759,0.1797,0.5614,14.2895,0.0407
4,Random Forest,0.50,0.8334,0.2185,0.4129,0.2857,0.7426,62.1842,2.9602
5,Random Forest SMOTE,0.15,0.7741,0.1817,0.5134,0.2684,0.7233,157.9591,3.2712
6,XGBoost,0.45,0.8385,0.2345,0.4419,0.3064,0.7582,5.6668,0.0558
7,XGBoost SMOTE,0.15,0.8295,0.2211,0.4407,0.2945,0.7482,27.3879,0.6405
8,LightGBM,0.60,0.8473,0.2393,0.4091,0.3019,0.7511,10.4578,1.0024
9,LightGBM SMOTE,0.15,0.8442,0.2383,0.4234,0.3050,0.7516,27.9315,0.8797


In [18]:
summary.to_csv('model_results/summary_results.csv')

In [19]:
joblib.dump(all_results, "model_results.pkl")

['model_results.pkl']

### Store confusion matrices

In [20]:
os.makedirs('model_results/confusion_matrices', exist_ok=True)

In [21]:
for r in all_results:

    cm = r["Confusion Matrix"]

    # Compute relative values:
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_norm,
        display_labels=["No Default", "Default"]
    )

    disp.plot(cmap="Blues", values_format=".2f")
    plt.title(r["Model"])
    plt.savefig(f"model_results/confusion_matrices/confusion_matrix_{r['Model'].replace(' ', '_')}.png",
                dpi=300, bbox_inches="tight")
    plt.close()